# VF-NeRF (conditional-NF fork) — Kaggle training

Trains (or restores) a **frozen nerfacto backbone** for every scene in `SCENES`
(cell 0; default `['bonsai', 'counter', 'kitchen', 'room']`, all Mip-NeRF 360),
then trains a **conditional Normalizing Flow** per scene that models
`P((3D point, 3D direction) | DINO feature)` against that scene's frozen NeRF,
and lets you **probe** every scene: pick points on each scene's training images
(or your own images -- run `python app/pick_points.py` locally to click them, or
hand-edit the per-scene lists in cell 6a), then generate the sampled novel views
for them in cell 6b. Cell 4 (a spiral flythrough render) is optional and
auto-skips a scene whose nerfacto was restored rather than trained.

### Before you run — REQUIRED (open the right sidebar → Settings)
1. **Accelerator → GPU T4 x2** (or **GPU P100**). Cell 0 hard-stops if this is off.
2. **Internet → On** — needs a phone-verified Kaggle account
   (Settings → Phone Verification). Cell 0 hard-stops if this is off.
3. Then **Save Version → Save & Run All (Commit)** for an unattended run — Kaggle
   allows up to 12 h and, unlike free Colab, will not reclaim the GPU mid-run.
   Everything written to `/kaggle/working/` is saved as the version's Output.

Heavy build artifacts (venv, repo, dataset) go in `/kaggle/temp/` (not persisted).
Checkpoints (+ any renders) land in `/kaggle/working/` as they are produced.

### Runtime and the 12 h cap
Per scene: ~1 h nerfacto (60k iters) if trained from scratch, plus ~45–90 min
conditional NF (30k steps, includes a ~6–12 min DINO precompute). Four scenes
from scratch is **~7–10 h** plus ~40 min env build + downloads -- close to the
cap. `bonsai`'s nerfacto is free (the repo ships a trained checkpoint). If a run
times out: **Save Version**, attach the partial `vf_nerf_outputs` as a Dataset
input (see "Loading an existing checkpoint" below), and **Run All** again --
any scene with a matching checkpoint is skipped, not retrained.

In [ ]:
# @title 0. Environment sanity check  (STOP if this cell raises)
import sys, platform, subprocess, os
print('system python:', sys.version)
print('platform:', platform.platform())
smi = subprocess.run(['bash','-c','nvidia-smi -L 2>/dev/null || true'], capture_output=True, text=True).stdout.strip()
print('GPU:', smi or '(none)')
if not smi:
    raise RuntimeError(
        'No GPU. Right sidebar -> Settings -> Accelerator -> GPU T4 x2 (or P100), '
        'and Internet -> On (needs a phone-verified account). Then re-run.')
net = subprocess.run(['bash','-c','curl -sI --max-time 10 https://pypi.org >/dev/null && echo ok || echo fail'], capture_output=True, text=True).stdout.strip()
print('internet:', net)
if net != 'ok':
    raise RuntimeError('No internet. Right sidebar -> Settings -> Internet -> On, then re-run.')
for d in ('/kaggle/temp', '/kaggle/working'):
    os.makedirs(d, exist_ok=True)


# Scenes to train + probe in this run (all Mip-NeRF 360, indoor/bounded -- see
# scripts/downloads/download_mipnerf360.py). Trim this list to train fewer.
SCENES = ['bonsai', 'counter', 'kitchen', 'room']
_ALLOWED_SCENES = ('bonsai', 'counter', 'kitchen', 'room')  # == downloader's --scene choices
assert SCENES and all(s in _ALLOWED_SCENES for s in SCENES), f'bad SCENES {SCENES}'
print('scenes:', SCENES)

In [ ]:
# @title 1. Build isolated CUDA 11.7 / Python 3.10 env + tiny-cuda-nn + repo
# Mirrors the debugged Colab setup: torch==1.13.1+cu117 has no cp311 wheel and
# tiny-cuda-nn needs the CUDA 11.7 dev headers, so we build a py3.10 venv rather
# than touch Kaggle's system interpreter.
setup = r'''#!/bin/bash
set -e
export DEBIAN_FRONTEND=noninteractive

TMP=/kaggle/temp
REPO=$TMP/VF-NeRF-conditional
VENV=$TMP/venv310
mkdir -p $TMP

echo '=== add NVIDIA CUDA apt repo (for the 11.7 dev packages) ==='
UBU=$(. /etc/os-release && echo ${VERSION_ID//./})   # 2204 / 2404
if ! ls /etc/apt/sources.list.d/ | grep -qi cuda; then
  wget -qO /tmp/cuda-keyring.deb https://developer.download.nvidia.com/compute/cuda/repos/ubuntu${UBU}/x86_64/cuda-keyring_1.1-1_all.deb || \
  wget -qO /tmp/cuda-keyring.deb https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb
  dpkg -i /tmp/cuda-keyring.deb
fi

echo '=== apt packages ==='
apt-get -qq update
# python3.10: default on 22.04; via deadsnakes otherwise
if ! apt-get -qq install -y python3.10 python3.10-venv python3.10-dev 2>/dev/null; then
  apt-get -qq install -y software-properties-common
  add-apt-repository -y ppa:deadsnakes/ppa
  apt-get -qq update
  apt-get -qq install -y python3.10 python3.10-venv python3.10-dev
fi
apt-get -q install -y \
    cuda-nvcc-11-7 cuda-cudart-dev-11-7 cuda-nvrtc-dev-11-7 libcublas-dev-11-7 libcufft-dev-11-7 \
    libcurand-dev-11-7 libcusolver-dev-11-7 libcusparse-dev-11-7 libnpp-dev-11-7 \
    libnvjpeg-dev-11-7 ninja-build ffmpeg

if [ ! -x /usr/local/cuda-11.7/bin/nvcc ]; then
  echo 'FATAL: /usr/local/cuda-11.7/bin/nvcc missing after apt install'
  ls -R /usr/local/cuda-11.7 2>/dev/null | head -40; apt-cache policy cuda-nvcc-11-7
  exit 1
fi

echo '=== register CUDA 11.7 lib path ==='
echo '/usr/local/cuda-11.7/lib64' > /etc/ld.so.conf.d/cuda-11-7.conf && ldconfig
# tiny-cuda-nn links against the CUDA driver lib (-lcuda). apt CUDA has only the
# stub; the real one ships with the GPU driver as libcuda.so.1 with no .so symlink.
STUB=/usr/local/cuda-11.7/lib64/stubs
REAL=$(ldconfig -p | awk '/libcuda\.so\.1/{print $NF; exit}')
[ -n "$REAL" ] && [ ! -e /usr/local/cuda-11.7/lib64/libcuda.so ] && ln -sf "$REAL" /usr/local/cuda-11.7/lib64/libcuda.so
export LIBRARY_PATH=/usr/local/cuda-11.7/lib64:$STUB${LIBRARY_PATH:+:$LIBRARY_PATH}
echo "libcuda: real=$REAL  stub=$(ls $STUB/libcuda.so 2>/dev/null)"

echo '=== venv310 ==='
[ -d $VENV ] || python3.10 -m venv $VENV
$VENV/bin/pip install -q 'setuptools<81' wheel

echo '=== torch 1.13.1+cu117 ==='
$VENV/bin/pip install -q torch==1.13.1 torchvision functorch --extra-index-url https://download.pytorch.org/whl/cu117
$VENV/bin/pip install -q ninja

echo '=== toolchain diagnostics ==='
export CUDA_HOME=/usr/local/cuda-11.7
export PATH=/usr/local/cuda-11.7/bin:$PATH
ls -d /usr/local/cuda* || true
which nvcc; nvcc --version 2>&1 | tail -2 || echo 'NO nvcc at /usr/local/cuda-11.7/bin'
gcc --version | head -1; g++ --version | head -1
# CUDA 11.7 nvcc rejects gcc>11; pin to gcc-11 if a newer default is present
if gcc -dumpversion | grep -qvE '^(9|10|11)'; then
  apt-get -qq install -y gcc-11 g++-11
  export CC=gcc-11 CXX=g++-11
  export NVCC_PREPEND_FLAGS='-ccbin g++-11'
  echo 'pinned to gcc-11'
fi

echo '=== build tiny-cuda-nn ==='
# derive the arch from the actual GPU (T4 -> 75, P100 -> 60, L4 -> 89, ...)
GPUCC=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d '. ')
export TCNN_CUDA_ARCHITECTURES=${GPUCC:-75}
echo "TCNN_CUDA_ARCHITECTURES=$TCNN_CUDA_ARCHITECTURES"
# master built fine on Colab recently; fall back through the last few tags if it
# has since moved past CUDA 11.7. Override the whole list with TCNN_REFS.
TCNN_OK=
for ref in ${TCNN_REFS:-master v1.6 v1.5}; do
  [ "$ref" = master ] && spec='git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch' \
                       || spec="git+https://github.com/NVlabs/tiny-cuda-nn/@${ref}#subdirectory=bindings/torch"
  echo "--- trying tiny-cuda-nn @ $ref ---" | tee -a /kaggle/working/tcnn_build.log
  if $VENV/bin/pip install --no-build-isolation -v "$spec" >> /kaggle/working/tcnn_build.log 2>&1; then
    TCNN_OK=$ref; break
  fi
  echo "  @ $ref failed:"; grep -E 'error:|fatal error|unsupported|cannot find -l|undefined reference|ld returned' /kaggle/working/tcnn_build.log | tail -8
done
if [ -z "$TCNN_OK" ]; then
  echo '!!! tiny-cuda-nn build FAILED for every ref — full log at /kaggle/working/tcnn_build.log'
  grep -nE 'error:|fatal error|unsupported|cannot find -l|undefined reference|ld returned' /kaggle/working/tcnn_build.log | tail -40
  exit 1
fi
echo "tiny-cuda-nn OK (@ $TCNN_OK)"
$VENV/bin/python -c 'import tinycudann as t; print("tcnn import ok", t.__version__ if hasattr(t,"__version__") else "")'

echo '=== clone repo ==='
rm -rf $REPO
git clone --quiet https://github.com/itayhanoch/VF-NeRF-conditional.git $REPO

echo '=== apply known repo fixes ==='
# 1) eval call site missing the `step` arg (crashes at the first full-image eval)
sed -i 's/metrics_dict, _ = self.model.get_image_metrics_and_images(outputs, batch)/metrics_dict, _ = self.model.get_image_metrics_and_images(outputs, batch, step)/' \
    $REPO/nerfstudio/pipelines/base_pipeline.py
# 2) scripts/render.py imports get_mask_from_view_likelihood, removed with the
#    registration pipeline — dead import, never used. Strip it so ns-render works.
sed -i '/get_mask_from_view_likelihood/d' $REPO/scripts/render.py
# 3) DINOv2 hub code (facebookresearch/dinov2 @ main) now needs torch>=2.0
#    (F.scaled_dot_product_attention). Prepend a math-identical fallback to the
#    one module that torch.hub.load's DINOv2 — nerfstudio/utils/dino_features.py.
$VENV/bin/python - "$REPO/nerfstudio/utils/dino_features.py" <<'PYEOF'
import sys, pathlib
p = pathlib.Path(sys.argv[1])
s = p.read_text()
if 'sdpa-shim' not in s:
    shim = ('\n# sdpa-shim (DINOv2 main needs torch>=2.0; this stack is torch 1.13)\n'
            'import math as _m\n'
            'import torch as _t\n'
            'import torch.nn.functional as _F\n'
            'if not hasattr(_F, "scaled_dot_product_attention"):\n'
            '    def _sdpa(q, k, v, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None):\n'
            '        sc = 1.0 / _m.sqrt(q.size(-1)) if scale is None else scale\n'
            '        a = _t.matmul(q, k.transpose(-2, -1)) * sc\n'
            '        if attn_mask is not None:\n'
            '            a = a.masked_fill(~attn_mask, float("-inf")) if attn_mask.dtype == _t.bool else a + attn_mask\n'
            '        a = a.softmax(-1)\n'
            '        if dropout_p:\n'
            '            a = _F.dropout(a, dropout_p)\n'
            '        return _t.matmul(a, v)\n'
            '    _F.scaled_dot_product_attention = _sdpa\n')
    # insert AFTER `from __future__` (must stay first statement); else after the docstring
    anchor = 'from __future__ import annotations\n'
    if anchor in s:
        s = s.replace(anchor, anchor + shim, 1)
    else:
        s = shim.lstrip() + '\n' + s
    p.write_text(s)
    print('dino_features.py: sdpa shim installed')
else:
    print('dino_features.py: sdpa shim already present')
PYEOF

echo '=== install repo + normalizing-flows ==='
cd $REPO
$VENV/bin/pip install -q --no-build-isolation -e . -e ./normalizing-flows

echo '=== SETUP COMPLETE ==='
'''
import os, subprocess
os.makedirs('/kaggle/temp', exist_ok=True)
with open('/kaggle/temp/setup.sh', 'w') as fh:
    fh.write(setup)
# pipefail so the cell sees bash's exit code, not tee's
rc = subprocess.call(
    ['bash', '-c', 'set -o pipefail; bash /kaggle/temp/setup.sh 2>&1 | tee /kaggle/working/setup.log'])
if rc != 0:
    print('--- last 60 lines of setup.log ---')
    print(subprocess.run(['tail', '-n', '60', '/kaggle/working/setup.log'], capture_output=True, text=True).stdout)
    raise RuntimeError(f'setup.sh failed (exit {rc}). See /kaggle/working/setup.log '
                       'and, for tiny-cuda-nn, /kaggle/working/tcnn_build.log')

In [ ]:
# @title 2. Config
import os, glob
TMP = '/kaggle/temp'
REPO = f'{TMP}/VF-NeRF-conditional'
VENV = f'{TMP}/venv310/bin'
WORK = '/kaggle/working'

# Per-scene paths (SCENES comes from cell 0).
SCENE_DATA = lambda s: f'{REPO}/data/mipnerf360/{s}'
COND_DIR   = lambda s: f'{WORK}/checkpoints/conditional_nf/{s}'

# nerfacto recipe mirrored from the original VF-NeRF (leosegre/VF_NeRF,
# reg_pipeline_pc.py): downscale 2, 60k iters, 1024 rays/batch, camera-opt off,
# full train split, center-method=focus, scene-scale 2. Applies to every scene.
NERFACTO_DOWNSCALE   = 2
NERFACTO_ITERS       = 60000
NERFACTO_RAYS        = 1024
NERFACTO_FORCE_RETRAIN = False   # True = ignore/delete any existing nerfacto run and retrain
COND_NF_MAX_STEPS    = 30000
COND_NF_FORCE_RETRAIN = False  # True = train even if a checkpoint is available for a scene
COND_NF_REDUCE_DIM = 6  # reduce the DINO condition to this many dims via a jointly-trained MLP before conditioning the flow (None = no reduction)
COND_NF_REDUCE_DIVIDE_FACTOR = 8  # only used when COND_NF_REDUCE_DIM is set
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-11.7/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')

NERF_OUTPUT_DIR = f'{WORK}/outputs'          # persisted; nerfacto nests <scene>/nerfacto/<ts>/
RENDER_DIR = f'{WORK}/renders'               # persisted
os.makedirs(RENDER_DIR, exist_ok=True)

# ---- per-scene checkpoint reuse: match by OUTPUT directory structure ----
# A scene is skipped (not retrained) if a matching checkpoint is found. Matching
# is structural -- the scene name must sit exactly where cell 5/7 themselves put
# it, mirroring the two layouts that actually occur (not a loose "scene name
# anywhere in the path" guess, which could mis-fire on e.g. a Dataset folder
# named "bathroom-photos"):
#   * nerfacto:       .../<scene>/nerfacto/<timestamp>/config.yml   (the raw
#                     nerfstudio output tree cell 5 trains into -- also what a
#                     *.tar.gz of outputs/ unpacks to)
#                     OR .../nerfacto_<scene>/config.yml             (cell 7's
#                     packaged bundle layout; also matches the repo's own
#                     checkpoints/nerfacto_<scene>/, so bonsai always resolves)
#   * conditional-NF: .../conditional_nf/<scene>/latest.pt (or cond_nf_step_*.pt)
#                     OR .../conditional_nf_<scene>/latest.pt (packaged bundle)
# Attach a Dataset (right sidebar -> Input -> Add Input) preserving one of these
# layouts intact for each scene you want to skip -- e.g. re-attach a prior run's
# vf_nerf_outputs.zip as-is (it already uses the nerfacto_<scene>/ and
# conditional_nf_<scene>/ bundle layout), or upload outputs/<scene>/ intact.
from pathlib import Path

def _nerf_scene_of(cfg_path):
    parts = Path(cfg_path).parts
    for i, seg in enumerate(parts):
        if seg == 'nerfacto' and i > 0 and parts[i - 1] in SCENES:
            return parts[i - 1]                       # .../<scene>/nerfacto/<ts>/config.yml
        if seg.startswith('nerfacto_') and seg[len('nerfacto_'):] in SCENES:
            return seg[len('nerfacto_'):]              # .../nerfacto_<scene>/config.yml
    return None

def _cond_scene_of(pt_path):
    parts = Path(pt_path).parts
    for i, seg in enumerate(parts):
        if seg == 'conditional_nf' and i + 1 < len(parts) and parts[i + 1] in SCENES:
            return parts[i + 1]                        # .../conditional_nf/<scene>/latest.pt
        if seg.startswith('conditional_nf_') and seg[len('conditional_nf_'):] in SCENES:
            return seg[len('conditional_nf_'):]         # .../conditional_nf_<scene>/latest.pt
    return None

PRETRAINED_NERF = {}            # scene -> config.yml (any attached Dataset OR the repo's own checkpoints/)
for c in (glob.glob('/kaggle/input/**/config.yml', recursive=True) +
          glob.glob(f'{REPO}/checkpoints/**/config.yml', recursive=True)):
    # ckpt may sit under nerfstudio_models/ (the usual nerfstudio layout, kept
    # intact by cell 7's packaging) or flat next to config.yml (the repo's own
    # committed checkpoints/nerfacto_bonsai/) -- accept either.
    if glob.glob(os.path.join(os.path.dirname(c), '**', '*.ckpt'), recursive=True):
        s = _nerf_scene_of(c)
        if s:
            PRETRAINED_NERF.setdefault(s, c)

INPUT_COND_PT = {}              # scene -> latest.pt / cond_nf_step_*.pt
for p in sorted(glob.glob('/kaggle/input/**/latest.pt', recursive=True) +
                glob.glob('/kaggle/input/**/cond_nf_step_*.pt', recursive=True)):
    s = _cond_scene_of(p)
    if s:
        INPUT_COND_PT.setdefault(s, p)

PRETRAINED_TARS = sorted(glob.glob('/kaggle/input/**/*.tar.gz', recursive=True) +
                         glob.glob('/kaggle/input/**/*.tgz', recursive=True))  # scene(s) inferred on extract, same structural rule

print('nerfacto restore :', PRETRAINED_NERF)
print('nerfacto tarballs:', PRETRAINED_TARS)
print('cond-NF restore  :', INPUT_COND_PT)

In [ ]:
# @title 2b. Download scenes + build downscaled images
import os, subprocess
from pathlib import Path
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

for s in SCENES:
    dd = SCENE_DATA(s)
    if not os.path.isdir(f'{dd}/images'):
        subprocess.run([f'{VENV}/python', 'scripts/downloads/download_mipnerf360.py',
                        '--scene', s, '--save-dir', f'{REPO}/data/mipnerf360'],
                       check=True, cwd=REPO)
    else:
        print(f'{dd}/images present, skip download')

    # nerfstudio 0.2.1 does NOT auto-generate images_<N>/; the dataparser just
    # looks for it. downscale 2 = training res (VF-NeRF recipe), 4 = spare.
    # Full res OOMs the image cache on a free-tier box.
    src = Path(dd) / 'images'
    n_src = len(list(src.glob('*')))
    for factor in sorted({NERFACTO_DOWNSCALE, 4}):
        dst = Path(dd) / f'images_{factor}'
        if dst.is_dir() and len(list(dst.glob('*'))) == n_src:
            print(f'  {s}/images_{factor} present, skip')
            continue
        dst.mkdir(exist_ok=True)
        def _f(p, factor=factor, dst=dst):
            im = Image.open(p); w, h = im.size
            im.resize((w // factor, h // factor), Image.LANCZOS).save(dst / p.name)
        with ThreadPoolExecutor(max_workers=8) as ex:
            list(ex.map(_f, sorted(src.glob('*'))))
        print(f'  {s}/images_{factor}: {len(list(dst.glob("*")))}')

In [ ]:
# @title 3. Train (or restore) the frozen nerfacto backbone -- per scene
import glob, subprocess, os, tarfile, re, shutil

def newest_config(s):
    # only a config whose run dir actually holds a loadable checkpoint counts --
    # this skips a half-trained run and, importantly, a stale malformed restore
    # dir (e.g. a bare `nerfacto_<scene>/` with no nerfstudio_models/), so the
    # restore branch below can rebuild it correctly.
    for pat in (f'{NERF_OUTPUT_DIR}/{s}/nerfacto/*/config.yml',
                f'{WORK}/**/{s}/nerfacto/*/config.yml'):
        c = sorted(p for p in glob.glob(pat, recursive=True)
                   if glob.glob(os.path.join(os.path.dirname(p), 'nerfstudio_models', '*.ckpt')))
        if c:
            return c[-1]
    return None

def repoint_output_dir(cfg_path, target=f'{WORK}/outputs'):
    # configs from another machine bake in an absolute output_dir; repoint it so
    # nerfstudio resolves <output_dir>/<exp>/nerfacto/<ts>/nerfstudio_models here.
    # nerfstudio serialises a Path as a multi-line !!python/object/apply:pathlib
    # .PosixPath + block-sequence-of-parts; a from-scratch config uses a plain
    # scalar. Pick the branch by which FORM is present -- not by whether re.sub
    # changed anything (a config already pointing here yields an identical sub).
    s = open(cfg_path).read()
    pathlib_pat = r'output_dir:(?: &\S+)? !!python/object/apply:pathlib\.PosixPath\n(?:- .*\n)+'
    block = ('output_dir: !!python/object/apply:pathlib.PosixPath\n'
             + ''.join(f'- {p}\n' for p in ['/'] + target.strip('/').split('/')))
    if re.search(pathlib_pat, s):
        s = re.sub(pathlib_pat, block, s, count=1)
    elif re.search(r'^output_dir: .+$', s, flags=re.M):
        s = re.sub(r'^output_dir: .+$', f'output_dir: {target}', s, count=1, flags=re.M)
    else:
        raise RuntimeError(f'no output_dir key found in {cfg_path}')
    open(cfg_path, 'w').write(s)

# Extract any attached tarballs once, up front -- each may contain one or more
# scenes' outputs/<scene>/nerfacto/<ts>/ trees; newest_config() below then finds
# whichever scenes they cover.
for tar in PRETRAINED_TARS:
    print('Extracting nerfacto tarball', tar)
    with tarfile.open(tar) as t:
        t.extractall(WORK)
for cfg in glob.glob(f'{WORK}/**/nerfacto/*/config.yml', recursive=True):
    repoint_output_dir(cfg)

NERF_CONFIGS, NERF_RESTORED = {}, {}
for s in SCENES:
    has_restore_src = (newest_config(s) is not None) or (s in PRETRAINED_NERF)
    if NERFACTO_FORCE_RETRAIN and not has_restore_src:
        shutil.rmtree(f'{NERF_OUTPUT_DIR}/{s}', ignore_errors=True)
        print(s, ': NERFACTO_FORCE_RETRAIN, cleared', f'{NERF_OUTPUT_DIR}/{s}')

    cfg = newest_config(s)

    if cfg is None and s in PRETRAINED_NERF:
        src_run = os.path.dirname(PRETRAINED_NERF[s])
        # nerfstudio's eval_setup recomputes the checkpoint dir from the config's
        # own output_dir/experiment_name/method_name/timestamp/relative_model_dir
        # -> the run directory MUST be named after the config's `timestamp`, and
        # the ckpt MUST sit under nerfstudio_models/. Neither holds for a packaged
        # `nerfacto_<scene>/` bundle or the repo's flat committed checkpoint, so
        # rebuild the canonical layout rather than copying the folder as-is.
        cfg_txt = open(f'{src_run}/config.yml').read()
        m = re.search(r'^timestamp:\s*(\S+)\s*$', cfg_txt, flags=re.M)
        ts = m.group(1) if m else 'restored'
        dst_run = f'{NERF_OUTPUT_DIR}/{s}/nerfacto/{ts}'
        print(s, ': restoring nerfacto from', src_run, '->', dst_run)
        os.makedirs(f'{dst_run}/nerfstudio_models', exist_ok=True)
        shutil.copy(f'{src_run}/config.yml', f'{dst_run}/config.yml')
        dpt = f'{src_run}/dataparser_transforms.json'
        if os.path.isfile(dpt):
            shutil.copy(dpt, f'{dst_run}/dataparser_transforms.json')
        ckpts = glob.glob(f'{src_run}/**/*.ckpt', recursive=True)
        assert ckpts, f'no *.ckpt found under {src_run}'
        for ck in ckpts:
            shutil.copy(ck, f'{dst_run}/nerfstudio_models/{os.path.basename(ck)}')
        repoint_output_dir(f'{dst_run}/config.yml')
        cfg = newest_config(s)

    NERF_RESTORED[s] = cfg is not None

    if cfg:
        print(s, ': using frozen nerfacto checkpoint', cfg)
    else:
        # nerfacto args mirrored from original VF-NeRF reg_pipeline_pc.py (minus the
        # stripped --nf-first-iter / --predict-view-likelihood which belonged to the
        # in-nerfacto NF that this fork replaced with a standalone trainer).
        cmd = [f'{VENV}/ns-train', 'nerfacto', '--data', SCENE_DATA(s),
               '--output-dir', NERF_OUTPUT_DIR, '--vis', 'tensorboard',
               '--viewer.quit-on-train-completion', 'True',
               '--max-num-iterations', str(NERFACTO_ITERS),
               '--pipeline.datamanager.train-num-rays-per-batch', str(NERFACTO_RAYS),
               '--pipeline.datamanager.camera-optimizer.mode', 'off',
               'nerfstudio-data',
               '--downscale-factor', str(NERFACTO_DOWNSCALE),
               '--center-method', 'focus',
               '--orientation-method', 'up',
               '--auto-scale-poses', 'True',
               '--scene-scale', '2']
        # NOTE: original VF-NeRF also passed --train-split-fraction 1.0, but it had
        # explicit train/eval transform files. With a single transforms.json, 1.0
        # leaves 0 eval cameras -> nerfacto's periodic eval AND ns-render (cell 4)
        # both crash. Keeping the 0.9 default (~90% train imgs) instead.
        print(s, ':', ' '.join(cmd)); subprocess.run(cmd, check=True, cwd=REPO)
        cfg = newest_config(s)

    assert cfg, f'no nerfacto config produced for {s}'
    # nerfstudio loads <dir(cfg)>/nerfstudio_models/step-*.ckpt -- and the dir
    # name must equal the config's timestamp (eval_setup rebuilds the path from
    # the config, it does not trust cfg's location). Verify both here.
    run_dir = os.path.dirname(cfg)
    ck = glob.glob(f'{run_dir}/nerfstudio_models/*.ckpt')
    ts_ok = re.search(r'^timestamp:\s*(\S+)\s*$', open(cfg).read(), flags=re.M)
    ts_ok = ts_ok and ts_ok.group(1) == os.path.basename(run_dir)
    assert ck and ts_ok, (
        f'{s}: bad nerfacto layout -- ckpt={bool(ck)}, dirname matches timestamp={bool(ts_ok)} '
        f'({run_dir})')
    print(s, ': NERF_CONFIG =', cfg)
    print(s, ': checkpoint  =', ck[-1])
    NERF_CONFIGS[s] = cfg

In [ ]:
# @title 4. (optional) Render a spiral flythrough of each frozen NeRF
# Auto-skipped per scene when nerfacto was restored from a checkpoint -- the
# flythrough is only useful to eyeball a fresh training run. Flip FORCE_RENDER
# to render every scene anyway.
import subprocess, glob
FORCE_RENDER = False

for s in SCENES:
    if NERF_RESTORED[s] and not FORCE_RENDER:
        print(s, ': nerfacto restored from a checkpoint -> skipping flythrough render.')
        continue
    out = f'{RENDER_DIR}/{s}_spiral.mp4'
    cmd = [f'{VENV}/ns-render', '--load-config', NERF_CONFIGS[s], '--traj', 'spiral',
           '--rendered-output-names', 'rgb', 'depth', '--output-path', out, '--seconds', '6']
    print(s, ':', ' '.join(cmd))
    r = subprocess.run(cmd, cwd=REPO)
    print(s, ': spiral render exit', r.returncode)

print('set FORCE_RENDER = True in this cell to render restored scenes anyway.')
print('renders present:', glob.glob(f'{RENDER_DIR}/*.mp4'))

In [ ]:
# @title 5. Train (or load) the conditional Normalizing Flow per scene -> /kaggle/working/checkpoints/
import subprocess, os, shutil

COND_LATEST = {}
for s in SCENES:
    os.makedirs(COND_DIR(s), exist_ok=True)
    COND_LATEST[s] = f'{COND_DIR(s)}/latest.pt'

    if INPUT_COND_PT.get(s) and not COND_NF_FORCE_RETRAIN:
        # train_conditional_nf.py has no resume flag, so a provided .pt is used as-is
        # (for the explorer / packaging); it is not continued. NB: a .pt trained
        # before the native-res DINO change is a different feature distribution --
        # set COND_NF_FORCE_RETRAIN = True in cell 2 to retrain instead.
        src = INPUT_COND_PT[s]
        print(s, ': using provided conditional-NF checkpoint', src)
        if os.path.abspath(src) != os.path.abspath(COND_LATEST[s]):
            shutil.copy(src, COND_LATEST[s])
        continue

    cmd = [f'{VENV}/python', '-u', 'scripts/train_conditional_nf.py',
           '--nerf-config', NERF_CONFIGS[s],
           '--scene-dir', SCENE_DATA(s),
           '--checkpoint-dir', COND_DIR(s),
           '--max-steps', str(COND_NF_MAX_STEPS),
           '--batch-size', '4096']
    if COND_NF_REDUCE_DIM is not None:
        cmd += ['--reduce-dim', str(COND_NF_REDUCE_DIM), '--reduce-divide-factor', str(COND_NF_REDUCE_DIVIDE_FACTOR)]
    print(s, ':', ' '.join(cmd))
    # -u + PYTHONUNBUFFERED so the 'step N/30000 | loss ...' lines stream live
    # rather than sitting in a block buffer for minutes. The first ~6-12 min is
    # a silent DINO precompute -- every training image at native resolution into
    # one memory-mapped dino_grids_*.npy (~1.7 GB) under SCENE_DATA(s)/dino_cache/
    # (a per-image progress line prints as it goes; re-runs reuse the file).
    # NB: the NF loss is a NEGATIVE log-likelihood of a CONTINUOUS density -- it
    # legitimately goes negative (e.g. ~ -4) as the flow sharpens. Watch that
    # the trend is downward and finite, not the sign.
    subprocess.run(cmd, check=True, cwd=REPO,
                   env={**os.environ, 'PYTHONUNBUFFERED': '1'})

for s in SCENES:
    print(f'--- {s} ---')
    !ls -la {COND_DIR(s)}

In [ ]:
# @title 6a. Pick points to probe (per scene)
# BEST: run  python app/pick_points.py --scene <name>  on your own machine -- click
# the objects you want to probe and paste the COORDS block it prints in below.
# (Kaggle's JupyterLab has no ipympl widget frontend, so there is no in-notebook
# click capture.)
# COORDS is a dict: one list of picks per scene in SCENES. Each pick is
# [ref, x, y] or [ref, x, y, backoff], x/y in that image's own pixels:
#   * ref = 'DSCF####.JPG'  -> a training frame of that scene (matched by filename
#     in images_2/)
#   * ref = 'myphoto.jpg'   -> any other image; attach a Kaggle Dataset containing
#     it (right sidebar -> Input -> Add Input). Only its DINO feature at the pixel
#     is used -- the rendered views are still that scene's renders.
#   * ref = int             -> old form: index into sorted(images_2/*)
# An optional 4th field sets the render-camera backoff (how far behind the sampled
# point the novel-view camera sits, in scene units):
#   * omitted or <= 0  -> auto: the frozen NeRF's depth along that pixel's real ray
#     for a training frame, else the scene-wide constant (printed by cell 6b).
#   * > 0              -> use that literal backoff for this probe (also the only way
#     to control standoff for an external image).
# Referencing frames by FILENAME (not position) keeps this aligned with the local
# picker (app/pick_points.py emits 3-field rows; add the 4th by hand). This cell
# shows each scene's GRID_VIEW full-size with a pixel grid, then redraws every
# pick as a numbered circle. It writes /kaggle/working/coords.json for cell 6b.
# A scene absent from COORDS (or mapped to an empty list) is skipped in 6b.
import glob, json, math, os
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from PIL import Image

COORDS = {
    'bonsai': [
        ['DSCF5577.JPG', 981, 329],   # bonsai frame 12
        ['DSCF5591.JPG', 205, 163],   # bonsai frame 26
        ['DSCF5602.JPG', 310, 241],   # bonsai frame 37
        ['DSCF5653.JPG', 486, 558],   # bonsai frame 88
        ['DSCF5659.JPG', 388, 448],   # bonsai frame 94
        ['DSCF5685.JPG', 229, 364],   # bonsai frame 120
        ['DSCF5715.JPG', 133, 638],   # bonsai frame 150
        ['DSCF5717.JPG', 197, 404],   # bonsai frame 152
        ['DSCF5732.JPG', 282, 170],   # bonsai frame 167
        ['DSCF5745.JPG', 110, 116],   # bonsai frame 180
        ['DSCF5745.JPG', 110, 727],   # bonsai frame 180
        ['white dor.jpeg', 605, 726],              # EXTERNAL -- attach a Kaggle Dataset containing 'white dor.jpeg'
        ['white dor.jpeg', 477, 583],              # EXTERNAL -- attach a Kaggle Dataset containing 'white dor.jpeg'
        ['organ.jpg', 685, 205],                   # EXTERNAL -- attach a Kaggle Dataset containing 'organ.jpg'
        ['herman miller chair.jpg', 407, 480],    # EXTERNAL -- attach a Kaggle Dataset containing 'herman miller chair.avif'
        ['herman miller chair.jpg', 180, 270],    # EXTERNAL -- attach a Kaggle Dataset containing 'herman miller chair.avif'
        ['cardboard box.jpg', 390, 252],           # EXTERNAL -- attach a Kaggle Dataset containing 'cardboard box.jpg'
        ['cardboard box.jpg', 385, 384],           # EXTERNAL -- attach a Kaggle Dataset containing 'cardboard box.jpg'
        ['bycicle.jpg', 1537, 275],                # EXTERNAL -- attach a Kaggle Dataset containing 'bycicle.jpg'
        ['bycicle.jpg', 1832, 997],                # EXTERNAL -- attach a Kaggle Dataset containing 'bycicle.jpg'
        ['bycicle.jpg', 1290, 1067],               # EXTERNAL -- attach a Kaggle Dataset containing 'bycicle.jpg'
        ['bycicle.jpg', 779, 282],                 # EXTERNAL -- attach a Kaggle Dataset containing 'bycicle.jpg'
        ['bonsai.jpg', 426, 415],                  # EXTERNAL -- attach a Kaggle Dataset containing 'bonsai.jpg'
        ['bonsai.jpg', 120, 425],                  # EXTERNAL -- attach a Kaggle Dataset containing 'bonsai.jpg'
        ['DSCF5845.JPG', 1515, 598],  # bonsai frame 280
        ['DSCF5844.JPG', 847, 28],    # bonsai frame 279
        ['DSCF5830.JPG', 222, 253],   # bonsai frame 265
        ['DSCF5819.JPG', 710, 258],   # bonsai frame 254
        ['DSCF5819.JPG', 663, 739],   # bonsai frame 254
        ['DSCF5819.JPG', 892, 444],   # bonsai frame 254
        ['DSCF5815.JPG', 585, 29],    # bonsai frame 250
    ],
    'counter': [
        ['DSCF5859.JPG', 1078, 289],  # counter frame 2
        ['DSCF5865.JPG', 752, 39],    # counter frame 8
        ['DSCF5865.JPG', 1472, 414],  # counter frame 8
        ['DSCF5878.JPG', 264, 82],    # counter frame 21
        ['DSCF5887.JPG', 603, 208],   # counter frame 30
        ['DSCF5895.JPG', 51, 279],    # counter frame 38
        ['DSCF5908.JPG', 412, 1016],  # counter frame 51
        ['DSCF5926.JPG', 464, 39],    # counter frame 69
        ['DSCF5930.JPG', 1465, 493],  # counter frame 73
        ['DSCF5938.JPG', 78, 446],    # counter frame 81
        ['DSCF5959.JPG', 1126, 713],  # counter frame 102
        ['DSCF5985.JPG', 138, 322],   # counter frame 128
        ['DSCF5981.JPG', 69, 343],    # counter frame 124
        ['DSCF6041.JPG', 38, 317],    # counter frame 184
        ['apples.jpeg', 2264, 1270],              # EXTERNAL -- attach a Kaggle Dataset containing 'apples.webp'
        ['apples.jpeg', 515, 1494],               # EXTERNAL -- attach a Kaggle Dataset containing 'apples.webp'
        ['cardboard box.jpg', 301, 272],          # EXTERNAL -- attach a Kaggle Dataset containing 'cardboard box.jpg'
        ['eggs carton.jpg', 752, 503],            # EXTERNAL -- attach a Kaggle Dataset containing 'eggs carton.jpg'
        ['green-pringles.png', 261, 170],         # EXTERNAL -- attach a Kaggle Dataset containing 'green-pringles.png'
        ['green-pringles.png', 226, 810],         # EXTERNAL -- attach a Kaggle Dataset containing 'green-pringles.png'
        ['metal bowe.jpg', 329, 394],            # EXTERNAL -- attach a Kaggle Dataset containing 'metal bowe.avif'
        ['silver refrigirator.jpg', 612, 982],    # EXTERNAL -- attach a Kaggle Dataset containing 'silver refrigirator.jpg'
    ],
    'kitchen': [
        ['DSCF0656.JPG', 1228, 569],  # kitchen frame 0
        ['DSCF0656.JPG', 645, 516],   # kitchen frame 0
        ['DSCF0657.JPG', 422, 54],    # kitchen frame 1
        ['DSCF0663.JPG', 971, 109],   # kitchen frame 7
        ['DSCF0689.JPG', 1377, 171],  # kitchen frame 33
        ['DSCF0709.JPG', 968, 693],   # kitchen frame 53
        ['DSCF0722.JPG', 274, 496],   # kitchen frame 66
        ['DSCF0743.JPG', 1060, 381],  # kitchen frame 87
        ['DSCF0770.JPG', 893, 225],   # kitchen frame 114
        ['DSCF0775.JPG', 652, 458],   # kitchen frame 119
        ['DSCF0775.JPG', 559, 266],   # kitchen frame 119
        ['DSCF0775.JPG', 932, 155],   # kitchen frame 119
        ['DSCF0784.JPG', 1196, 634],  # kitchen frame 128
        ['DSCF0793.JPG', 1271, 137],  # kitchen frame 137
        ['DSCF0807.JPG', 93, 208],    # kitchen frame 151
    ],
    'room': [
        ['DSCF4667.JPG', 1285, 85],   # room frame 0
        ['DSCF4667.JPG', 28, 604],    # room frame 0
        ['DSCF4667.JPG', 728, 238],   # room frame 0
        ['DSCF4689.JPG', 1523, 291],  # room frame 22
        ['DSCF4689.JPG', 985, 76],    # room frame 22
        ['DSCF4697.JPG', 407, 89],    # room frame 30
        ['DSCF4713.JPG', 162, 641],   # room frame 46
        ['DSCF4753.JPG', 461, 468],   # room frame 86
        ['DSCF4756.JPG', 192, 490],   # room frame 89
        ['DSCF4760.JPG', 745, 304],   # room frame 93
        ['DSCF4760.JPG', 127, 524],   # room frame 93
        ['DSCF4754.JPG', 985, 23],    # room frame 87
        ['DSCF4767.JPG', 207, 804],   # room frame 100
        ['DSCF4767.JPG', 48, 911],    # room frame 100
        ['DSCF4839.JPG', 1351, 306],  # room frame 172
        ['DSCF4840.JPG', 1415, 177],  # room frame 173
        ['DSCF4967.JPG', 1507, 868],  # room frame 300
        ['metal bowe.jpg', 580, 381],             # EXTERNAL -- attach a Kaggle Dataset containing 'metal bowe.avif'
        ['home theater system.jpg', 424, 380],     # EXTERNAL -- attach a Kaggle Dataset containing 'home theater system.jpg'
        ['home theater system.jpg', 1263, 409],    # EXTERNAL -- attach a Kaggle Dataset containing 'home theater system.jpg'
        ['piano.jpeg', 789, 1031],                 # EXTERNAL -- attach a Kaggle Dataset containing 'piano.jpeg'
        ['television.jpg', 110, 78],               # EXTERNAL -- attach a Kaggle Dataset containing 'television.jpg'
        ['television.jpg', 440, 45],               # EXTERNAL -- attach a Kaggle Dataset containing 'television.jpg'
        ['white dor.jpeg', 603, 618],              # EXTERNAL -- attach a Kaggle Dataset containing 'white dor.jpeg'
        ['white dor.jpeg', 470, 547],              # EXTERNAL -- attach a Kaggle Dataset containing 'white dor.jpeg'
    ],
}
GRID_VIEW = {s: 0 for s in SCENES}   # frame to show large with a coordinate grid, per scene

_DS = globals().get('NERFACTO_DOWNSCALE', 2)   # same folder pick_points.py / cell 6b use

def _circle(ax, x, y, k, wh):
    ax.add_patch(Circle((x, y), radius=max(wh) // 45, fill=False, color='red', lw=2))
    ax.text(x + 12, y - 12, str(k), color='red', fontsize=13, weight='bold')

json.dump({s: [list(c) for c in COORDS.get(s, [])] for s in SCENES},
          open(f'{WORK}/coords.json', 'w'))
print(f'wrote {WORK}/coords.json')

for s in SCENES:
    cs = COORDS.get(s, [])
    if not cs:
        print(f'--- {s}: no COORDS, skipping preview (and cell 6b) ---')
        continue

    _imgs = sorted(glob.glob(f'{SCENE_DATA(s)}/images_{_DS}/*'))
    assert _imgs, f'no images in {SCENE_DATA(s)}/images_{_DS}/ -- run cell 2b'
    _by_name = {os.path.basename(p): p for p in _imgs}

    def _resolve(ref, _imgs=_imgs, _by_name=_by_name, _scene=s):
        if isinstance(ref, int):
            assert 0 <= ref < len(_imgs), f'frame index {ref} out of range 0..{len(_imgs) - 1}'
            return _imgs[ref]
        if os.path.basename(ref) in _by_name:
            return _by_name[os.path.basename(ref)]
        hits = sorted(glob.glob(f'/kaggle/input/**/{os.path.basename(ref)}', recursive=True))
        if not hits:
            print(f'  ! {ref!r} is not a {_scene} frame and not under /kaggle/input --',
                  'attach a Kaggle Dataset containing it (cell 6b will fail without it)')
        return hits[0] if hits else None

    print(f'--- {s}: {len(_imgs)} frames ---')
    for _c in cs:
        _bk = _c[3] if len(_c) > 3 else -1
        _mode = f'manual {_bk}' if _bk > 0 else 'auto backoff'
        print('  ', _c[:3], f'[{_mode}]', '->', _resolve(_c[0]))

    _gv = GRID_VIEW.get(s, 0)
    _gp = _resolve(_gv)
    _im = Image.open(_gp); _w, _h = _im.size
    print(f'  grid view: {os.path.basename(str(_gp))} (W,H)=({_w},{_h})')
    plt.figure(figsize=(15, 10))
    plt.imshow(_im)
    plt.xticks(range(0, _w, 100)); plt.yticks(range(0, _h, 100))
    plt.grid(True, color='cyan', alpha=0.4, lw=0.5)
    for _k, _c in enumerate(cs):
        _ref, _x, _y = _c[0], _c[1], _c[2]
        if _resolve(_ref) == _gp:
            _circle(plt.gca(), _x, _y, _k, (_w, _h))
    plt.title(f'{s}: {os.path.basename(str(_gp))} -- read (x, y) off the grid; circles = COORDS here')
    plt.show()

    _seen, _panels = set(), []
    for _c in cs:
        _p = _resolve(_c[0])
        if _p and _p not in _seen:
            _seen.add(_p); _panels.append((_c[0], _p))
    if _panels:
        _ncol = min(len(_panels), 4)
        _nrow = math.ceil(len(_panels) / _ncol)
        _fig, _ax = plt.subplots(_nrow, _ncol, figsize=(5 * _ncol, 4 * _nrow), squeeze=False)
        for _slot in range(_nrow * _ncol):
            _a = _ax[_slot // _ncol][_slot % _ncol]; _a.axis('off')
            if _slot >= len(_panels):
                continue
            _ref, _path = _panels[_slot]
            _pim = Image.open(_path); _pw, _ph = _pim.size
            _a.imshow(_pim); _a.set_title(f'{s}: {os.path.basename(str(_path))}')
            for _k, _c in enumerate(cs):
                if _resolve(_c[0]) == _path:
                    _circle(_a, _c[1], _c[2], _k, (_pw, _ph))
        plt.show()

In [ ]:
EXPLORER_SRC = r'''"""Headless point-and-generate explorer for the conditional-NF fork.

This is the committed-run ("Save & Run All") path for probing the conditional NF:
it reproduces app/gradio_app.py's workflow without a UI. For each (ref, x, y)
probe -- ref = a dataset frame index or an external image's basename -- take that
pixel's DINOv2 patch-token feature as the NF condition, sample candidate (point,
direction) pairs, rank by likelihood, render the top few through the frozen NeRF,
and write a montage PNG. (Points come from the notebook's cell 6a; run
app/gradio_app.py directly on a local GPU box for a live click UI.)

Runs inside the py3.10 venv (needs torch 1.13 + nerfstudio + tinycudann); the
notebook cell that launches it displays the PNGs from the base kernel.
"""
import argparse
import glob
import json
import math
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle
import torch

REPO = "/kaggle/temp/VF-NeRF-conditional"
sys.path.insert(0, REPO)

from nerfstudio.cameras.cameras import Cameras
from nerfstudio.fields.nf_field import ConditionalNFField
from nerfstudio.utils.dino_features import DinoExtractor, load_image_chw_01
from nerfstudio.utils.eval_utils import eval_setup

# nerfstudio's default post-auto-orient world-up axis (orientation-method "up").
DEFAULT_WORLD_UP = torch.tensor([0.0, 0.0, 1.0])


# --- helpers copied from app/gradio_app.py (kept gradio-free) ------------------

def default_backoff_distance(cameras: Cameras) -> float:
    centers = cameras.camera_to_worlds[..., :3, 3]
    scene_center = centers.mean(dim=0)
    return (centers - scene_center).norm(dim=-1).median().item()


def build_camera_from_point_direction(position, direction, reference_cameras, backoff_distance,
                                      world_up=DEFAULT_WORLD_UP):
    device = position.device
    world_up = world_up.to(device)
    forward = direction / direction.norm().clamp_min(1e-8)
    up_ref = world_up
    if torch.abs(torch.dot(forward, up_ref)) > 0.99:
        up_ref = torch.tensor([1.0, 0.0, 0.0], device=device)
    right = torch.cross(forward, up_ref)
    right = right / right.norm().clamp_min(1e-8)
    up = torch.cross(right, forward)
    camera_origin = position - forward * backoff_distance
    rotation = torch.stack([right, up, -forward], dim=-1)
    c2w = torch.cat([rotation, camera_origin.unsqueeze(-1)], dim=-1)
    return Cameras(
        camera_to_worlds=c2w.unsqueeze(0),
        fx=reference_cameras.fx[0:1], fy=reference_cameras.fy[0:1],
        cx=reference_cameras.cx[0:1], cy=reference_cameras.cy[0:1],
        width=reference_cameras.width[0:1], height=reference_cameras.height[0:1],
        camera_type=reference_cameras.camera_type[0:1],
    ).to(device)


def load_conditional_nf(checkpoint_path: Path, device: torch.device):
    ckpt = torch.load(checkpoint_path, map_location=device)
    field = ConditionalNFField(
        context_dim=ckpt["context_dim"],
        num_dims=ckpt.get("num_dims", 6),
        num_blocks=ckpt["num_blocks"],
        hidden_dim=ckpt["hidden_dim"],
        cond_prior=ckpt["cond_prior"],
        use_cond_in_coupling=True,
        use_batchnorm=ckpt["use_batchnorm"],
        reduce_dim=ckpt.get("reduce_dim"),
        reduce_divide_factor=ckpt.get("reduce_divide_factor", 8),
        device=str(device),
    )
    field.load_state_dict(ckpt["model_state"])
    field.eval()
    return field, ckpt["dino_model_name"], int(ckpt.get("step", -1))


# --- main --------------------------------------------------------------------

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--nerf-config", required=True)
    ap.add_argument("--scene-dir", required=True)
    ap.add_argument("--cond-nf-checkpoint", required=True)
    ap.add_argument("--probes", required=True,
                    help='JSON list of [ref, x, y] or [ref, x, y, backoff]; ref = int frame '
                         'index or "name.ext" external image. backoff <= 0 (or omitted) = auto '
                         '(NeRF depth for dataset frames, scene constant otherwise); > 0 = that literal.')
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--downscale", type=int, default=2,
                    help="images_<N>/ folder the coords were picked on (must match cell 6a / pick_points.py)")
    ap.add_argument("--num-samples", type=int, default=200)
    ap.add_argument("--render-top", type=int, default=5)
    ap.add_argument("--render-downscale", type=float, default=3.0)
    args = ap.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"Loading frozen NeRF from {args.nerf_config} ...", flush=True)
    config, pipeline, _, _ = eval_setup(Path(args.nerf_config), test_mode="inference")
    nerf_model = pipeline.model.to(device).eval()
    for p in nerf_model.parameters():
        p.requires_grad_(False)

    # Build the reference cameras in the SAME frame the frozen NeRF (and hence the
    # conditional NF) was trained in -- the checkpoint's own dataparser config,
    # not stock defaults (center-method / scene-scale differ).
    dp = config.pipeline.datamanager.dataparser
    dp.data = Path(args.scene_dir)
    outs = dp.setup().get_dataparser_outputs(split="train")
    cameras = outs.cameras.to(device)
    # dataparser train-split order (matches `cameras`), NOT sorted(glob(...)) order.
    cam_by_name = {Path(p).name: i for i, p in enumerate(outs.image_filenames)}
    const_backoff = default_backoff_distance(cameras)

    # Image list + coordinate space = sorted(glob("images_<downscale>/*")), the
    # EXACT folder cell 6a and pick_points.py pick on (--downscale is passed from
    # cell 6b, not guessed from the nerfacto config, so it can't drift). The
    # conditional NF was also trained on DINO features from this same folder.
    img_dir = Path(args.scene_dir) / f"images_{args.downscale}"
    image_filenames = sorted(img_dir.glob("*"))
    assert image_filenames, f"no images in {img_dir} -- run cell 1 (or fix --downscale)"
    print(f"{len(image_filenames)} images in {img_dir} | const backoff {const_backoff:.3f}", flush=True)

    field, dino_model_name, step = load_conditional_nf(Path(args.cond_nf_checkpoint), device)
    print(f"conditional-NF checkpoint step = {step}", flush=True)
    extractor = DinoExtractor(model_name=dino_model_name, device=str(device))

    grid_cache = {}

    by_name = {p.name: p for p in image_filenames}

    def resolve_ref(ref):
        """ref -> image path.
          * int / digit string -> dataset frame by position (old form)
          * a training frame's basename -> that file in images_<ds>/
          * any other basename / path -> an external image: a real path if it
            exists, else looked up under /kaggle/input/** (attach a Dataset)."""
        if isinstance(ref, int) or (isinstance(ref, str) and ref.lstrip("-").isdigit()):
            i = int(ref)
            assert 0 <= i < len(image_filenames), (
                f"frame index {i} out of range 0..{len(image_filenames) - 1}")
            return image_filenames[i]
        p = Path(ref)
        if p.name in by_name:
            return by_name[p.name]
        if p.is_file():
            return p
        hits = sorted(glob.glob(f"/kaggle/input/**/{p.name}", recursive=True))
        if not hits:
            raise RuntimeError(
                f"image {ref!r} not found -- it is not a training frame (images/{p.name}) "
                f"and not under /kaggle/input/**; attach a Kaggle Dataset that contains "
                f"{p.name!r} (right sidebar -> Input -> Add Input)")
        return Path(hits[0])

    def patch_feature(ref, x, y):
        """Pixel (x, y) on image `ref` -> its DINOv2 patch-token feature.

        Indexes the patch grid directly (same as the NF trainer's sample_batch);
        no giant per-pixel upsampled map.
        """
        if ref not in grid_cache:
            img = load_image_chw_01(resolve_ref(ref))
            with torch.no_grad():
                g, (h, w) = extractor.extract_patch_grid(img)
            grid_cache[ref] = (g.float(), img, h, w)
        g, img, h, w = grid_cache[ref]
        hp, wp = g.shape[-2:]
        py = min(max(int(y * hp / h), 0), hp - 1)
        px = min(max(int(x * wp / w), 0), wp - 1)
        return g[:, py, px].reshape(-1).to(device), img

    def probe_backoff(ref, x, y, img_h, img_w):
        """Camera standoff for rendering this probe's candidates.

        A dataset frame has a real camera ray through pixel (x, y); the frozen
        NeRF's median termination depth along it is the true camera-to-surface
        distance -- a far better render backoff than the scene-wide constant.
        External images (and held-out frames) have no camera in this frame -> the
        constant. Returns (distance, "depth" | "const").
        """
        if isinstance(ref, int) or (isinstance(ref, str) and ref.lstrip("-").isdigit()):
            name = image_filenames[int(ref)].name
        else:
            name = Path(ref).name
        i = cam_by_name.get(name)
        if i is None:
            return const_backoff, "const"
        cy = y * float(cameras.height[i]) / img_h
        cx = x * float(cameras.width[i]) / img_w
        rb = cameras.generate_rays(
            camera_indices=torch.tensor([[i]]),
            coords=torch.tensor([[cy, cx]], dtype=torch.float32),
        ).to(device)
        with torch.no_grad():
            t = float(nerf_model(rb)["depth"].reshape(-1)[0])
        if not math.isfinite(t) or t <= 1e-4:
            print(f"  ! depth render at ({x},{y}) = {t!r}; using constant backoff", flush=True)
            return const_backoff, "const"
        return t, "depth"

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    probes = json.loads(args.probes)
    manifest = []

    for k, entry in enumerate(probes):
        ref, x, y = entry[0], int(entry[1]), int(entry[2])
        depth_override = float(entry[3]) if len(entry) > 3 else -1.0
        src = resolve_ref(ref)
        cond, img = patch_feature(ref, x, y)
        _H, _W = int(img.shape[-2]), int(img.shape[-1])
        if depth_override > 0:
            bo, bo_src = depth_override, "manual"
        else:
            bo, bo_src = probe_backoff(ref, x, y, _H, _W)
        print(f"probe {k}: {ref} @ ({x},{y})  <- {src}  ({_W}x{_H})  backoff {bo:.3f} ({bo_src})", flush=True)
        if not (0 <= x < _W and 0 <= y < _H):
            print(f"  ! ({x},{y}) is OUTSIDE this {_W}x{_H} image -- coords were picked "
                  f"on a different resolution or a different frame", flush=True)
        with torch.no_grad():
            samples = field.sample(num_samples=args.num_samples, context=cond)
            logp = field.log_prob(
                samples, cond.unsqueeze(0).expand(args.num_samples, -1)
            ).squeeze(-1)
        order = torch.argsort(logp, descending=True)[: args.render_top]
        n = len(order)

        fig, ax = plt.subplots(1, n + 1, figsize=(4 * (n + 1), 4))
        ax = [ax] if n == 0 else list(ax)
        ax[0].imshow(img.permute(1, 2, 0).numpy())
        ax[0].add_patch(Circle((x, y), radius=max(img.shape[-2:]) / 45,
                               fill=False, color="red", lw=2.5))
        ax[0].set_title(f"{ref} @ ({x},{y})\nbackoff {bo:.3f} ({bo_src})")
        ax[0].axis("off")
        for j, idx in enumerate(order):
            cam = build_camera_from_point_direction(
                samples[idx, :3], samples[idx, 3:], cameras, bo
            )
            cam.rescale_output_resolution(1.0 / args.render_downscale)
            cx, cy = float(cam.cx.squeeze()), float(cam.cy.squeeze())
            with torch.no_grad():
                o = nerf_model.get_outputs_for_camera_ray_bundle(
                    cam.generate_rays(camera_indices=0)
                )
            ax[j + 1].imshow(o["rgb"].clamp(0, 1).cpu().numpy())
            ax[j + 1].add_patch(Rectangle((cx - 16, cy - 16), 32, 32,
                                           fill=False, color="red", lw=2))
            ax[j + 1].set_title(f"logp {logp[idx].item():.2f}")
            ax[j + 1].axis("off")

        png = out_dir / f"probe_{k:02d}.png"
        fig.tight_layout()
        fig.savefig(png, dpi=90)
        plt.close(fig)
        manifest.append(str(png))
        print(f"probe {k}: {ref} ({x},{y}) -> {png}", flush=True)

    print("MANIFEST " + json.dumps(manifest), flush=True)


if __name__ == "__main__":
    main()
'''

In [ ]:
# @title 6b. Generate novel views for the picked points (per scene)
# For each scene with COORDS entries: for each point, take that pixel's DINOv2
# patch feature as the NF condition, draw NUM_SAMPLES candidate (position,
# direction) 6-D samples from that scene's flow, score each with the flow's
# log_prob, and render the RENDER_TOP HIGHEST-likelihood ones through that
# scene's frozen NeRF into a montage (circled source point + those top views).
# A COORDS ref is a training frame's filename, an external image's filename
# (resolved under /kaggle/input/**), or an int index. Re-run after editing 6a;
# works in a committed run too. The log and each montage's left panel show
# 'backoff <v> (depth|const|manual)' -- the render-camera standoff: NeRF depth
# for a dataset-frame pixel, that scene's constant (also printed per scene)
# otherwise, or a COORDS 4th-field override.
import subprocess, os, json, glob
from IPython.display import Image as _Img, display

COORDS = json.load(open(f'{WORK}/coords.json'))
assert any(COORDS.get(s) for s in SCENES), 'no points -- run cell 6a first'
print('points:', COORDS)
NUM_SAMPLES, RENDER_TOP, RENDER_DOWNSCALE = 200, 5, 3.0

subprocess.run([f'{VENV}/pip', 'install', '-q', 'matplotlib'], check=True)
with open('/kaggle/temp/explorer.py', 'w') as fh:
    fh.write(EXPLORER_SRC)

EXPLORE_DIR = f'{WORK}/explorer'
os.makedirs(EXPLORE_DIR, exist_ok=True)

for s in SCENES:
    probes = COORDS.get(s, [])
    if not probes:
        print(f'--- {s}: no COORDS, skipping ---')
        continue
    out_dir = f'{EXPLORE_DIR}/{s}'
    os.makedirs(out_dir, exist_ok=True)
    for _p in glob.glob(f'{out_dir}/probe_*.png'):
        os.remove(_p)
    cmd = [f'{VENV}/python', '-u', '/kaggle/temp/explorer.py',
           '--nerf-config', NERF_CONFIGS[s], '--scene-dir', SCENE_DATA(s),
           '--cond-nf-checkpoint', COND_LATEST[s],
           '--probes', json.dumps(probes), '--out-dir', out_dir,
           '--downscale', str(NERFACTO_DOWNSCALE),
           '--num-samples', str(NUM_SAMPLES), '--render-top', str(RENDER_TOP),
           '--render-downscale', str(RENDER_DOWNSCALE)]
    print(f'--- {s} ---'); print(' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=REPO,
                   env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for png in sorted(glob.glob(f'{out_dir}/probe_*.png')):
        print(png)
        display(_Img(filename=png))

In [ ]:
# @title 7. Package outputs (all scenes)
import shutil, os, glob
bundle = f'{WORK}/vf_nerf_outputs'
shutil.rmtree(bundle, ignore_errors=True)
os.makedirs(bundle, exist_ok=True)
for s in SCENES:
    run_dir = os.path.dirname(NERF_CONFIGS[s])
    shutil.copytree(run_dir, f'{bundle}/nerfacto_{s}',
                    ignore=shutil.ignore_patterns('*.tfevents*'))
    shutil.copytree(COND_DIR(s), f'{bundle}/conditional_nf_{s}')
    probe_pngs = glob.glob(f'{WORK}/explorer/{s}/probe_*.png')
    if probe_pngs:
        os.makedirs(f'{bundle}/probes_{s}', exist_ok=True)
        for png in probe_pngs:
            shutil.copy(png, f'{bundle}/probes_{s}')
for mp4 in glob.glob(f'{RENDER_DIR}/*.mp4'):
    shutil.copy(mp4, bundle)
shutil.make_archive(bundle, 'zip', bundle)
print('bundle:', bundle + '.zip')
!du -sh {bundle}.zip {WORK}/renders {WORK}/checkpoints {WORK}/outputs
!find {bundle} -maxdepth 3 -type f | sort

## Getting your results

**Committed run (Save & Run All):** when the version finishes, open it and use the
**Output** tab — download `vf_nerf_outputs.zip` (per scene: nerfacto checkpoint +
config, conditional-NF `latest.pt`, probe montages) + the flythrough videos, or
the individual files.

**Interactive run:** the file browser on the right shows `/kaggle/working/` — right-click
→ Download. Do a **Save Version** first so the outputs are also stored server-side.

If the GUI file browser / Output tab is unavailable, get a direct download link from a cell:

```python
from IPython.display import FileLink
import shutil; shutil.make_archive('/kaggle/working/vf_nerf_outputs', 'zip', '/kaggle/working/vf_nerf_outputs')
FileLink('vf_nerf_outputs.zip')   # click it (works while the session is alive)
```

## Loading an existing checkpoint into this notebook

Kaggle notebooks can only read outside files through an attached **Dataset** (or Secret).
There is no Drive mount. So:

1. **kaggle.com -> Create -> New Dataset.** Upload your checkpoint file(s). The scene
   is inferred from *directory structure*, matching exactly what cells 5/7 themselves
   produce -- not just any folder/filename that happens to mention the scene:
   - nerfacto: a path ending `.../<scene>/nerfacto/<timestamp>/config.yml` (the raw
     nerfstudio output tree), **or** `.../nerfacto_<scene>/config.yml` (the packaged
     bundle layout below).
   - conditional-NF: `.../conditional_nf/<scene>/latest.pt`, **or**
     `.../conditional_nf_<scene>/latest.pt` (packaged bundle layout).
   A prior run's `vf_nerf_outputs.zip` already uses the bundle layout
   (`nerfacto_<scene>/`, `conditional_nf_<scene>/`), so it can be re-attached as-is to
   resume every scene it contains at once -- just don't rename or flatten those folders
   before uploading:
   - a **nerfacto** backup: either a `*.tar.gz` of an `outputs/` tree (what the earlier
     Colab `tar_ckpts.py` produced), **or** just the loose `nerfacto/<timestamp>/` run
     dir (`config.yml` + `nerfstudio_models/*.ckpt` + `dataparser_transforms.json`).
     Either works -- Kaggle auto-extracts an uploaded archive, so a `.tar.gz` ends up
     as the loose tree anyway. **or**
   - a **conditional-NF** checkpoint: `latest.pt` or `cond_nf_step_*.pt` from
     `scripts/train_conditional_nf.py`.
   Set it Private if you like; give it any title.
2. In this notebook: right sidebar -> **Input -> Add Input** -> your dataset -> **Add**.
   It mounts read-only at `/kaggle/input/<dataset-slug>/`.
3. Re-run from **cell 2**. It auto-detects the files per scene:
   - nerfacto tarball or loose run dir found -> **cell 3 restores whichever scenes it
     can match by name**, repoints their baked-in `output_dir`, and skips their
     training;
   - `.pt` found -> **cell 5 copies it to that scene's `latest.pt` and skips its NF
     training** (the trainer has no resume flag, so a `.pt` is only *used*, not
     continued).

Every scene in `SCENES` (cell 0) still downloads + gets its downscaled images built in
**cell 2b** either way (needed for DINO features, the render camera path, and the
dataparser the config points at) -- restoring a checkpoint skips training, not data prep.

**External probe images** ride the same mechanism: put any images in a Dataset, attach
it, and reference them by filename in cell 6a's per-scene `COORDS` (e.g.
`["myphoto.jpg", 640, 480]`). Cell 6b resolves the name under `/kaggle/input/**`,
conditions the flow on that pixel's DINO feature, and renders the sampled views through
that scene's frozen NeRF.

To get files *out* to Google Drive there's no built-in mount — download
`vf_nerf_outputs.zip` and upload it yourself, or add an `rclone`/Drive-API cell backed by
a Kaggle Secret.

## Interactive viewing — what works where

| tool | what it shows | Kaggle | needs |
|---|---|---|---|
| cell 4 video | spiral flythrough of the reconstructed NeRF | ✅ committed or interactive — **optional**, auto-skipped per scene when nerfacto is restored (set `FORCE_RENDER=True`) | nothing extra |
| `app/pick_points.py` + cells 6a/6b | click points on the images locally -> paste the `COORDS` block into 6a -> NF samples novel-view rays -> render top few per point (montage PNGs, 6b) | ✅ committed **or** interactive (picker runs on your machine) | both checkpoints, per scene |
| `ns-viewer` | free 3D navigation of the NeRF | ❌ not from Kaggle | browser must reach `ws://localhost:<port>` — i.e. local machine, or SSH port-forward from a rented GPU box (RunPod/Vast/Lambda) |

So: for a quick look at a scene's NeRF, force the cell 4 spiral render. To probe a
scene's conditional NF, run `python app/pick_points.py --scene <name>` on your own
machine -- it opens that scene's training images, you click the objects to probe, and it
prints a `COORDS` block to paste into **cell 6a** (under that scene's key); then run
**cell 6b**. (Kaggle's JupyterLab has no `ipympl` widget frontend, so there is no
in-notebook click capture; you can also hand-edit `COORDS` in 6a, which shows a pixel
grid per scene and draws your picks as circles.) Full free-fly `ns-viewer` needs a
machine you can point a browser at `localhost` on (local, or `ssh -L 7007:localhost:7007`
into a rented GPU box) — not Kaggle or Colab.